# 随机森林 SHAP 与 FastForest 的耗时对比

同一棵随机森林、同一批要解释的样本，比较：

1. `treeshap`：`shap.TreeExplainer`，sklearn 随机森林只走单核。
A Unified Approach to Interpreting Model Predictions (SHAP，NIPS 2017)
2. `fasttreeshap`：`fast_tree_shap`，LinkedIn FastTreeSHAP v2，多核。（Fast TreeSHAP，NeurIPS 2021 XAI4Debugging Workshop）
https://github.com/linkedin/fasttreeshap
https://doi.org/10.48550/arXiv.2109.09847

3. `fastforest`：ThrunGroup FastForest，MABSplit 加速的是训练，不是 TreeSHAP。（MABSplit: Faster Forest Training Using Multi-Armed Bandits，NeurIPS 2022）
https://github.com/ThrunGroup/FastForest

4.Linear tree shap (NeurIPS 2022)
https://github.com/yupbank/linear_tree_shap

前两项只比解释时间，不把训练时间算进去。第三项单独比训练时间，和上面已经拟合好的 sklearn 随机森林用同一批训练样本。内核选 **Python (esmvaltool)**。先不要改大下面的样本量，确认能跑通后再加大 `n_time`、`n_lat`、`n_lon` 或 `n_estimators`。

FastForest 已克隆到 `/Users/huangfeini/Documents/FastForest`。笔记本里用 `sys.path` 指向这个目录，不要安装它的 `requirements.txt`。


In [1]:
import sys
from pathlib import Path

fastforest_root = Path("/Users/huangfeini/Documents/FastForest")
if str(fastforest_root) not in sys.path:
    sys.path.insert(0, str(fastforest_root))
print(fastforest_root, "on sys.path:", str(fastforest_root) in sys.path)

/Users/huangfeini/Documents/FastForest on sys.path: True


In [2]:
from pathlib import Path
import sys
import time

root = Path.cwd().resolve()
if (root / "src" / "rf_shap").exists():
    sys.path.insert(0, str(root / "src"))
elif (root.parent / "src" / "rf_shap").exists():
    sys.path.insert(0, str(root.parent / "src"))
fastforest_root = Path("/Users/huangfeini/Documents/FastForest")
if fastforest_root.exists() and str(fastforest_root) not in sys.path:
    sys.path.insert(0, str(fastforest_root))

import numpy as np
import pandas as pd

from rf_shap.cube.tabular import cube_to_table
from rf_shap.explain.shap_explain import compute_shap
from rf_shap.models.registry import build_model
from rf_shap.models.split import split_table
from rf_shap.synthetic import make_synthetic_cubes


In [3]:
N_TIME, N_LAT, N_LON = 12, 8, 10
N_ESTIMATORS, MAX_DEPTH = 40, 8
MAX_SAMPLES = 400
SEED = 0

x, y = make_synthetic_cubes(n_time=N_TIME, n_lat=N_LAT, n_lon=N_LON, seed=SEED)
table = cube_to_table(x, y)
split = split_table(table, {"method": "random", "test_size": 0.3}, seed=SEED)
train, test = split.train, split.test

t0 = time.perf_counter()
model = build_model(
    "rf",
    {"n_estimators": N_ESTIMATORS, "max_depth": MAX_DEPTH, "n_jobs": -1},
    seed=SEED,
)
model.fit(train.X, train.y)
fit_seconds = time.perf_counter() - t0
print(f"训练样本 {len(train.y)}，测试样本 {len(test.y)}，特征 {train.X.shape[1]}")
print(f"随机森林拟合 {fit_seconds:.2f} 秒（不计入下面的 SHAP 对比）")


训练样本 672，测试样本 288，特征 3
随机森林拟合 0.07 秒（不计入下面的 SHAP 对比）


In [4]:
def time_shap(method, algorithm="v2"):
    t0 = time.perf_counter()
    result = compute_shap(
        model,
        test,
        method=method,
        algorithm=algorithm,
        max_samples=MAX_SAMPLES,
        n_jobs=-1,
        seed=SEED,
        feature_perturbation="tree_path_dependent",
    )
    seconds = time.perf_counter() - t0
    return result, seconds


tree_result, tree_seconds = time_shap("treeshap")
fast_result, fast_seconds = time_shap("fasttreeshap", algorithm="v2")

timing = pd.DataFrame(
    [
        {"方法": "treeshap（单核 TreeSHAP）", "秒": tree_seconds, "解释样本数": len(tree_result.sample_positions)},
        {"方法": "fasttreeshap v2（多核）", "秒": fast_seconds, "解释样本数": len(fast_result.sample_positions)},
    ]
)
timing["相对 treeshap 的加速"] = timing["秒"].iloc[0] / timing["秒"]
timing


/opt/miniconda3/envs/esmvaltool/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
IPython could not be loaded!


,方法,秒,解释样本数,相对 treeshap 的加速
0,treeshap（单核 TreeSHAP）,0.576911,288,1.000000
1,fasttreeshap v2（多核）,0.044724,288,12.899414


In [5]:
same_rows = np.array_equal(tree_result.sample_positions, fast_result.sample_positions)
corr = np.corrcoef(tree_result.values.ravel(), fast_result.values.ravel())[0, 1]
print(f"两次解释的是同一批样本: {same_rows}")
print(f"两套 SHAP 值的相关系数: {corr:.4f}")
print(f"fasttreeshap 记录的算法: {fast_result.algorithm}")


两次解释的是同一批样本: True
两套 SHAP 值的相关系数: 1.0000
fasttreeshap 记录的算法: v2


In [6]:
from rf_shap.models.fast_forest import fast_forest
from rf_shap.predict.metrics import regression_metrics

fit_timing = None
try:
    fast_model = fast_forest(
        n_estimators=N_ESTIMATORS,
        max_depth=MAX_DEPTH,
        solver="MAB",
        random_state=SEED,
    )
except ImportError as exc:
    print(exc)
else:
    t0 = time.perf_counter()
    fast_model.fit(train.X, train.y)
    fast_fit_seconds = time.perf_counter() - t0
    fast_pred = fast_model.predict(test.X)
    sklearn_pred = model.predict(test.X)
    fit_timing = pd.DataFrame(
        [
            {
                "方法": "sklearn 随机森林",
                "训练秒": fit_seconds,
                "测试 R2": regression_metrics(test.y, sklearn_pred)["r2"],
            },
            {
                "方法": "fastforest（MABSplit）",
                "训练秒": fast_fit_seconds,
                "测试 R2": regression_metrics(test.y, fast_pred)["r2"],
            },
        ]
    )
    fit_timing["相对 sklearn 的加速"] = fit_timing["训练秒"].iloc[0] / fit_timing["训练秒"]
fit_timing

,方法,训练秒,测试 R2,相对 sklearn 的加速
0,sklearn 随机森林,0.066782,0.994455,1.000000
1,fastforest（MABSplit）,20.534792,0.993001,0.003252
